# Spark Structured Streaming


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql import functions as F
from config import TOPIC_FATIGA


spark = (
    SparkSession.builder
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.13:3.5.0')  # Conector Kafka
    .master('local[*]')
    .appName('SportsSoccerTelemetry')
    .getOrCreate()
)

In [16]:
# TODO: Define schema for Kafka messages
schema = StructType([
    StructField("schema_version", StringType(), True),
    StructField("player_id", StringType(), True),
    StructField("event_id", StringType(), True),
    StructField("match_id", IntegerType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", IntegerType(), True),
    StructField("bpm", IntegerType(), True),
    StructField("pos_x", DoubleType(), True),
    StructField("pos_y", DoubleType(), True),
    StructField("speed_kmh", DoubleType(), True),
    StructField("acceleration_ms2", DoubleType(), True)
])

In [17]:
# TODO: Define the topic name for Kafka producer
KAFKA_TOPIC = TOPIC_FATIGA

In [18]:
df = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', "localhost:9092")
    .option('subscribe', KAFKA_TOPIC)                     # Kafka topic to subscribe to
    .option('startingOffsets', 'earliest')                # Read from the beginning of the topic
    .load()
)

In [22]:
# TODO: Parse message with defined schema
events = (
    df.select(F.from_json(F.col("value").cast("string"), schema).alias("e"))
    .select("e.*")
)
#Comprobar que el parseo está bien hecho
events.printSchema()

root
 |-- schema_version: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: integer (nullable = true)
 |-- bpm: integer (nullable = true)
 |-- pos_x: double (nullable = true)
 |-- pos_y: double (nullable = true)
 |-- speed_kmh: double (nullable = true)
 |-- acceleration_ms2: double (nullable = true)



In [23]:
# TODO: Define aggregations by player within the window

In [24]:
# TODO: Write the output to console for debugging purposes